# Testing on PISA dataset with randomized missing data, all 3 missing data handling options tested on the same random values

1. Importing package and dataset

In [39]:
!pip install iita_python
!git clone https://gist.github.com/717f0147675b0c8ed25e50d583c943bf.git

import numpy as np
import iita_python as iita
import iita_python.fit_metrics as iita_fm
from iita_python.additional_ce import AdditionalCEDataset

from iita_python.utils import read_rp
from random import randint, choice, shuffle

fatal: destination path '717f0147675b0c8ed25e50d583c943bf' already exists and is not an empty directory.


2. Testing function

In [45]:
def add_missing_values(dataset, skips, choicePool=[i for i in range(5)]):    
    new_dataset = dataset.copy()

    for _ in range(skips):
        while (True):
            a = choice(choicePool)
            b = randint(0, new_dataset.shape[0] - 1)
            if (not (np.isnan(new_dataset.loc[b, a]) or (np.nansum(new_dataset.to_numpy(), axis=0)[a] == 1) or (np.nansum(new_dataset.to_numpy(), axis=1)[b] == 1))):
                break
        new_dataset.loc[b, a] = np.nan
    
    return new_dataset

In [50]:
def get_all_ces(rp):
    normal_data = iita.Dataset(rp)
    add_ce_data = AdditionalCEDataset(rp)

    return [normal_data.ce, normal_data.relative_ce, add_ce_data.missing_value_substitution_ce()]

In [51]:
def test(skips, calc_count=3, calculator=get_all_ces, bias=[1,1,1,1,1]):
    correct = [True for _ in range(calc_count)]
    any_correct = calc_count
    correct_qo = None
    correct_count = [0 for _ in range(calc_count)]
    missing_amount = 0
    data = read_rp('./717f0147675b0c8ed25e50d583c943bf/pisa.csv')

    choicePool = []
    items = list(range(data.shape[1]))
    shuffle(items)

    for i, item in enumerate(items):
        for _ in range(bias[i]):
            choicePool.append(item)

    while (any_correct and missing_amount < data.shape[0]*data.shape[1] - 10):
        print(missing_amount, correct)
        ces = calculator(data)

        test_dataset = iita.Dataset(data)

        unfolded_ces = [iita.unfold_examples(ce) for ce in ces]

        all_qos = [iita.ind_gen(unf_ce, test_dataset.items) for unf_ce in unfolded_ces]

        for (i, qos) in enumerate(all_qos):
            if (not correct[i]):
                continue

            best_qo_diff = float('inf')

            for j, qo in enumerate(qos):
                qo_diff = iita_fm.mini_iita_fit(test_dataset, qo)
                if (qo_diff < best_qo_diff):
                    best_qo_diff = qo_diff
                    best_qo_id = j

            best_qo = sorted([(int(a), int(b)) for a, b in all_qos[i][best_qo_id].get_edge_list()])
            if (correct_qo is None):
                correct_qo = best_qo

            if (best_qo != correct_qo):
                correct[i] = False
                any_correct -= 1
            else:
                correct_count[i] += skips

        missing_amount += skips
        data = add_missing_values(data, skips, choicePool)

    return correct_count

In [48]:
def iter_test(skips, iters, **kwargs):
  iter_res = []
  for i in range(iters):
    print(f'ITER {i}')

    res = test(skips, **kwargs)
    print(res)
    iter_res.append(res)
  return iter_res

3. Running the tests

In [ ]:
iters = 2000 #amount of iterations to do
skips = 17 #amount of missing values to add at a time

res = iter_test(skips, iters)

4. Analyzing the tests

In [ ]:
data = read_rp('./717f0147675b0c8ed25e50d583c943bf/pisa.csv')
res = (np.array(res) / (data.shape[0] * data.shape[1])).round(3)

In [ ]:
import csv
with open('sim_missing_data_results.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['normal', 'relative', 'item_mean_substitution_CE'])
    writer.writerows(res)

4.1. Average

In [ ]:
np.mean(res).round(3)

np.float64(0.25)

4.2. Standard deviation

In [ ]:
np.std(res).round(3)

np.float64(0.089)

5. Testing function for biased item choice

In [ ]:
iters = 2000 # amount of iterations to do
skips = 17 # amount of missing values to add at a time
bias = [15, 11, 5, 3, 2] # probability distribution across the items. higher = more often

res_biased = iter_test(skips, iters, bias=bias)

In [ ]:
res_biased = (np.array(res_biased) / (data.shape[0] * data.shape[1])).round(3)

In [ ]:
np.mean(res_biased).round(3)

np.float64(0.126)

In [ ]:
np.std(res_biased).round(3)

np.float64(0.055)